In [1]:
import os
import numpy as np
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from vllm import LLM, SamplingParams


# ============================================================
# 知识库文档与测试集（不变）
# ============================================================
documents = [
    "深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。",
    "反向传播是训练神经网络的核心算法，通过链式法则计算梯度来更新权重。",
    "卷积神经网络（CNN）擅长处理图像数据，通过卷积核提取局部特征。",
    "循环神经网络（RNN）用于处理序列数据，但存在梯度消失问题。",
    "Transformer 架构基于自注意力机制，完全抛弃了循环结构，已成为大语言模型的基础。",
    "GPT 系列模型是 OpenAI 开发的生成式预训练 Transformer，用于文本生成。",
    "Qwen 是阿里巴巴通义千问团队开发的大语言模型，支持中英文对话。",
    "vLLM 是一个高性能的 LLM 推理引擎，使用 PagedAttention 管理 KV 缓存。",
    "RAG（检索增强生成）结合了信息检索和文本生成，能提升答案的事实准确性。",
    "向量数据库如 Faiss 和 Milvus 用于高效存储和检索高维嵌入向量。",
    "激活函数如 ReLU 和 GELU 为神经网络引入非线性，使其能学习复杂映射。",
    "Adam 优化器结合了动量和自适应学习率，是深度学习中最常用的优化算法之一。",
]

test_dataset = [
    {
        "question": "什么是深度学习？",
        "ground_truth": "深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。",
        "relevant_docs": [0]
    },
    {
        "question": "Transformer 架构的核心是什么？",
        "ground_truth": "Transformer 架构基于自注意力机制，完全抛弃了循环结构。",
        "relevant_docs": [4]
    },
    {
        "question": "vLLM 是什么？",
        "ground_truth": "vLLM 是一个高性能的 LLM 推理引擎，使用 PagedAttention 管理 KV 缓存。",
        "relevant_docs": [7]
    },
    {
        "question": "什么是 RAG？",
        "ground_truth": "RAG（检索增强生成）结合了信息检索和文本生成，能提升答案的事实准确性。",
        "relevant_docs": [8]
    },
    {
        "question": "Adam 优化器有什么特点？",
        "ground_truth": "Adam 优化器结合了动量和自适应学习率，是深度学习中最常用的优化算法之一。",
        "relevant_docs": [11]
    },
    {
        "question": "深度学习如何训练网络？",
        "ground_truth": "使用反向传播算法，通过链式法则计算梯度来更新权重。",
        "relevant_docs": [0, 1]
    },
]

# ============================================================
# 加载模型
# ============================================================
print("加载嵌入模型...")
embedding_llm = LLM(
    model="intfloat/e5-small",
    task="embed",
    enforce_eager=True
)

print("加载生成模型...")
gen_llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_memory_utilization=0.5,
    max_model_len=1024,
    enforce_eager=True
)


INFO 07-23 16:56:05 __init__.py:190] Automatically detected platform cuda.
加载嵌入模型...
INFO 07-23 16:56:08 config.py:2382] Downcasting torch.float32 to torch.float16.
WARNING 07-23 16:56:16 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-23 16:56:16 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-23 16:56:16 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='intfloat/e5-small', speculative_config=None, tokenizer='intfloat/e5-small', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-23 16:56:26 model_runner.py:1115] Loading model weights took 0.0633 GB
加载生成模型...
INFO 07-23 16:56:38 config.py:542] This model supports multiple tasks: {'score', 'generate', 'reward', 'embed', 'classify'}. Defaulting to 'generate'.
WARNING 07-23 16:56:38 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-23 16:56:38 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-23 16:56:39 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disabl

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-23 16:56:42 model_runner.py:1115] Loading model weights took 2.8870 GB
INFO 07-23 16:56:43 worker.py:267] Memory profiling takes 0.66 seconds
INFO 07-23 16:56:43 worker.py:267] the current vLLM instance can use total_gpu_memory (11.76GiB) x gpu_memory_utilization (0.50) = 5.88GiB
INFO 07-23 16:56:43 worker.py:267] model weights take 2.89GiB; non_torch_memory takes 0.02GiB; PyTorch activation peak memory takes 1.39GiB; the rest of the memory reserved for KV Cache is 1.58GiB.
INFO 07-23 16:56:43 executor_base.py:110] # CUDA blocks: 3686, # CPU blocks: 9362
INFO 07-23 16:56:43 executor_base.py:115] Maximum concurrency for 1024 tokens per request: 57.59x
INFO 07-23 16:56:46 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 4.06 seconds


In [2]:

# ============================================================
# 生成文档向量 ———— 直接传字符串列表！
# ============================================================
print("生成文档向量...")
doc_texts = [f"passage: {doc}" for doc in documents]
doc_outputs = embedding_llm.embed(doc_texts)

doc_embeddings = []
for output in doc_outputs:
    emb = output.outputs.embedding
    if isinstance(emb, list):
        doc_embeddings.append(emb)
    else:
        doc_embeddings.append(list(emb))
doc_embeddings = np.array(doc_embeddings)
print(f"文档向量维度: {doc_embeddings.shape}")


生成文档向量...


Processed prompts: 100%|██████████| 12/12 [00:00<00:00, 49.13it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

文档向量维度: (12, 384)


In [3]:
sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256
    )

# ============================================================
# 检索函数 ———— 直接传字符串！
# ============================================================
def retrieve(query, k=3):
    query_output = embedding_llm.embed([f"query: {query}"])
    raw_emb = query_output[0].outputs.embedding
    if isinstance(raw_emb, list):
        query_embedding = np.array(raw_emb)
    else:
        query_embedding = np.array(list(raw_emb))

    # 余弦相似度
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    doc_norms = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    similarities = np.dot(query_norm, doc_norms.T)

    top_k_indices = np.argsort(similarities)[::-1][:k]
    return top_k_indices.tolist(), similarities[top_k_indices].tolist()


# ============================================================
# 生成函数
# ============================================================
def generate_rag_answer(query, retrieved_indices):
    context = "\n".join([f"- {documents[i]}" for i in retrieved_indices])
    rag_prompt = f"""根据参考资料回答问题，如果无法回答请说明。

【参考资料】
{context}

【问题】
{query}

【回答】"""

    output = gen_llm.generate([rag_prompt], sampling_params)
    return output[0].outputs[0].text


def generate_direct_answer(query):
    direct_prompt = f"请回答问题：{query}"
    output = gen_llm.generate([direct_prompt], sampling_params)
    return output[0].outputs[0].text


# ============================================================
# 评估指标（不变）
# ============================================================
def evaluate_retrieval(retrieved_indices, relevant_indices):
    k = len(retrieved_indices)
    retrieved_set = set(retrieved_indices)
    relevant_set = set(relevant_indices)

    true_positives = len(retrieved_set & relevant_set)
    recall = true_positives / len(relevant_set) if len(relevant_set) > 0 else 0
    precision = true_positives / k if k > 0 else 0
    return recall, precision


def simple_text_similarity(text1, text2):
    import re

    def tokenize(text):
        tokens = re.findall(r'[\u4e00-\u9fff]|[a-zA-Z]+|\d+', text.lower())
        return set(tokens)

    tokens1 = tokenize(text1)
    tokens2 = tokenize(text2)

    if not tokens1 or not tokens2:
        return 0.0

    intersection = tokens1 & tokens2
    union = tokens1 | tokens2
    jaccard = len(intersection) / len(union)

    key_terms = {t for t in tokens2 if len(t) >= 2}
    if not key_terms:
        return jaccard

    key_hit = len(key_terms & tokens1) / len(key_terms)
    return 0.6 * jaccard + 0.4 * key_hit



In [6]:

# ============================================================
# 运行评估
# ============================================================
print("\n" + "=" * 70)
print("RAG 系统离线评估")
print("=" * 70)

rag_retrieval_recalls = []
rag_retrieval_precisions = []
rag_answer_scores = []
direct_answer_scores = []

for i, item in enumerate(test_dataset):
    question = item["question"]
    ground_truth = item["ground_truth"]
    relevant_docs = item["relevant_docs"]

    print(f"\n--- 测试 {i+1}: {question} ---")

    retrieved_indices, scores = retrieve(question, k=5)
    recall, precision = evaluate_retrieval(retrieved_indices, relevant_docs)
    rag_retrieval_recalls.append(recall)
    rag_retrieval_precisions.append(precision)

    print(f"  检索文档索引: {retrieved_indices} (期望: {relevant_docs})")
    print(f"  Recall@{len(retrieved_indices)}: {recall:.2f}")
    print(f"  Precision@{len(retrieved_indices)}: {precision:.2f}")

    rag_answer = generate_rag_answer(question, retrieved_indices)
    rag_score = simple_text_similarity(rag_answer, ground_truth)
    rag_answer_scores.append(rag_score)
    print(f"  RAG 相似度: {rag_score:.3f}")
    print(f"  RAG 回答: {rag_answer[:100]}...")

    direct_answer = generate_direct_answer(question)
    direct_score = simple_text_similarity(direct_answer, ground_truth)
    direct_answer_scores.append(direct_score)
    print(f"  直接回答相似度: {direct_score:.3f}")
    print(f"  直接回答: {direct_answer[:100]}...")

# ============================================================
# 汇总报告
# ============================================================
print("\n\n" + "=" * 70)
print("评估结果汇总")
print("=" * 70)

print(f"\n[检索质量]")
print(f"  平均 Recall:   {np.mean(rag_retrieval_recalls):.3f}")
print(f"  平均 Precision: {np.mean(rag_retrieval_precisions):.3f}")

print(f"\n[生成质量]")
print(f"  RAG 平均相似度:     {np.mean(rag_answer_scores):.3f}")
print(f"  直接回答平均相似度:  {np.mean(direct_answer_scores):.3f}")
print(f"  RAG 提升:           {(np.mean(rag_answer_scores) - np.mean(direct_answer_scores)):.3f}")



RAG 系统离线评估

--- 测试 1: 什么是深度学习？ ---


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 52.28it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


  检索文档索引: [0, 1, 10, 4, 9] (期望: [0])
  Recall@5: 1.00
  Precision@5: 0.20


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.17s/it, est. speed input: 35.95 toks/s, output: 61.35 toks/s]


  RAG 相似度: 0.300
  RAG 回答: 
深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。它通过反向传播算法训练神经网络，通过链式法则计算梯度来更新权重。激活函数如 ReLU 和 GELU 为神经网络引入非线性，使其能...


...

  检索文档索引: [0, 1, 10, 9, 4] (期望: [0, 1])
  Recall@5: 1.00
  Precision@5: 0.40


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it, est. speed input: 81.46 toks/s, output: 60.02 toks/s]


  RAG 相似度: 0.202
  RAG 回答: 
深度学习通过使用多层神经网络来学习数据的层次化表示。训练这些网络的核心算法是反向传播。反向传播通过链式法则计算梯度来更新网络的权重。激活函数如ReLU和GELU在神经网络中引入了非线性，使网络能够学...


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.17s/it, est. speed input: 2.40 toks/s, output: 61.47 toks/s]

  直接回答相似度: 0.093
  直接回答:  深度学习是一种机器学习方法，它模仿人脑的神经元结构，通过构建多层的神经网络，从数据中学习特征和模式。以下是深度学习训练网络的基本步骤：

1. 数据准备：首先，需要收集和准备大量的数据。这些数据可以...


评估结果汇总

[检索质量]
  平均 Recall:   1.000
  平均 Precision: 0.233

[生成质量]
  RAG 平均相似度:     0.440
  直接回答平均相似度:  0.303
  RAG 提升:           0.136


In [7]:

print(f"\n[详细对比表]")
print(f"{'Question':^5} {'RAG Similarity':^20} {'Direct Similarity':^20} {'Improvement':^15}")
print("-" * 85)
for i, item in enumerate(test_dataset):
    q = i
    r_score = rag_answer_scores[i]
    d_score = direct_answer_scores[i]
    imp = r_score - d_score
    print(f"{q:^5} {r_score:^20.3f} {d_score:^20.3f} {imp:^15.3f}")

print("\n\n实验完成！")


[详细对比表]
Question    RAG Similarity     Direct Similarity     Improvement  
-------------------------------------------------------------------------------------
  0          0.300                0.174              0.126     
  1          0.514                0.441              0.073     
  2          0.505                0.136              0.369     
  3          0.501                0.472              0.029     
  4          0.615                0.503              0.113     
  5          0.202                0.093              0.109     


实验完成！
